# 04 — Time Series Forecasting

**Input:** `data/processed/hour_clean.csv` — aggregated to **daily** totals

**Goals:**
- Resample hourly data to daily demand
- Decompose time series (trend, seasonality, residual)
- Test for stationarity (ADF test)
- Fit SARIMA model
- Fit Prophet model
- Compare models with MAE / RMSE / MAPE

**Key concept:** forecasting models use only information available *up to* the forecast origin — never look ahead.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from prophet import Prophet
import sys
sys.path.append("..")
from src.utils import regression_metrics
import warnings
warnings.filterwarnings("ignore")

plt.style.use("seaborn-v0_8-whitegrid")

## 1. Resample to daily

In [ ]:
df = pd.read_csv("../data/processed/hour_clean.csv", parse_dates=["dteday"])
daily = df.groupby("dteday")["cnt"].sum().reset_index()
daily.columns = ["ds", "y"]
daily = daily.set_index("ds").sort_index()
print(daily.shape)
daily.head()

In [ ]:
daily["y"].plot(figsize=(14, 4), title="Daily bike rentals")
plt.tight_layout()

## 2. Train / test split (last 90 days as test)

In [ ]:
cutoff = daily.index.max() - pd.DateOffset(days=90)
train_ts = daily[daily.index <= cutoff]
test_ts  = daily[daily.index >  cutoff]
print(f"Train: {len(train_ts)} days | Test: {len(test_ts)} days")

## 3. Decomposition

In [ ]:
# TODO: decompose using both 'additive' and 'multiplicative' — which fits better?
# period=7 for weekly seasonality
result = seasonal_decompose(train_ts["y"], model="additive", period=7)
result.plot()
plt.tight_layout()

## 4. Stationarity — ADF test

In [ ]:
# TODO: test for stationarity; if non-stationary, try differencing
def adf_report(series, label=""):
    stat, p, *_ = adfuller(series.dropna())
    print(f"{label:30s} ADF stat: {stat:.3f}  p-value: {p:.4f}  → {'STATIONARY' if p < 0.05 else 'NON-STATIONARY'}")

adf_report(train_ts["y"], "Level")
adf_report(train_ts["y"].diff(), "First difference")
adf_report(train_ts["y"].diff(7), "Weekly difference")

## 5. SARIMA

In [ ]:
# TODO: tune (p,d,q)(P,D,Q,s) — start simple, check ACF/PACF plots for guidance
# Hint: from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
model_sarima = SARIMAX(
    train_ts["y"],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False
)
res_sarima = model_sarima.fit(disp=False)
print(res_sarima.summary())

In [ ]:
pred_sarima = res_sarima.forecast(steps=len(test_ts))
pred_sarima.index = test_ts.index
regression_metrics(test_ts["y"].values, pred_sarima.values, "SARIMA")

## 6. Prophet

In [ ]:
# Prophet expects columns 'ds' and 'y'
train_prophet = train_ts.reset_index().rename(columns={"ds": "ds", "y": "y"})

m = Prophet(weekly_seasonality=True, yearly_seasonality=True, daily_seasonality=False)
m.fit(train_prophet)

future = m.make_future_dataframe(periods=len(test_ts))
forecast = m.predict(future)
pred_prophet = forecast.set_index("ds")["yhat"].loc[test_ts.index]

regression_metrics(test_ts["y"].values, pred_prophet.values, "Prophet")

## 7. Visual comparison

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(test_ts.index, test_ts["y"], label="Actual", lw=2)
plt.plot(test_ts.index, pred_sarima.values, label="SARIMA", ls="--")
plt.plot(test_ts.index, pred_prophet.values, label="Prophet", ls=":")
plt.legend()
plt.title("Forecast vs Actual — test period")
plt.tight_layout()

## 8. Reflect

- Which model wins on each metric? Why might that be?
- What could improve SARIMA? (better order selection, exogenous variables)
- What could improve Prophet? (holidays, custom seasonalities)
- How would you productionise a forecasting pipeline?